In [1]:
# load data
import pandas as pd
eb_geo_stanza_df = pd.read_json('geotagged_articles_stanza_df.json', orient='records', lines=True)
eb_geo_flair_df = pd.read_json('geotagged_articles_flair_df.json', orient='records', lines=True)
eb_geo_spacy_df = pd.read_json('geotagged_articles_spacy_df.json', orient='records', lines=True)
eb_geo_eg_df = pd.read_json('articles_with_locations_eg_df.json', orient='records', lines=True)
len(eb_geo_stanza_df)

17270

In [2]:
# load annotated samples
eb_annotated_samples = pd.read_json('eb_geo_samples_annotated.json', orient='records', lines=True)
len(eb_annotated_samples)

89

In [28]:
eb_annotated_samples.columns

Index(['term_uri', 'name', 'hq_text', 'is_location', 'locations'], dtype='object')

In [6]:
def get_all_locations_for_sample(eb_geo_sample):
    """
    This function gets all the locations for a given eb sample, which includes the term name if it is identified as location, and values from 'locations' list the sample.
    :param eb_geo_sample: dict object with 'term_uri', 'name', 'hq_text', 'is_location', 'locations'
    :return: a list of locations
    """
    locations = []
    if eb_geo_sample['is_location']:
        locations.append({
            "name": eb_geo_sample['name'],
            "start": 0,
            "end": 0
        })
    for location in eb_geo_sample['locations']:
        location_name = location['name']
        if "start" in location:
            location_start = location['start']
        else:
            location_start = location["start_index"]

        if "end" in location:
            location_end = location['end']
        else:
            location_end = location["end_index"]
        locations.append({
            "name": location_name,
            "start": location_start,
            "end": location_end
        })
    return locations


In [96]:
eb_annotated_samples["all_locations"] = eb_annotated_samples.apply(get_all_locations_for_sample, axis=1)
eb_annotated_samples.iloc[1]

term_uri         https://w3id.org/hto/ArticleTermRecord/9910796...
name                                                       KASIMOW
hq_text          a city of the Russian province Riasan, the cap...
is_location                                                   True
locations        [{'name': 'Riasan', 'start': 31, 'end': 37}, {...
last-checked                                                   NaN
all_locations    [{'name': 'KASIMOW', 'start': 0, 'end': 0}, {'...
Name: 1, dtype: object

In [97]:
eb_annotated_locations = eb_annotated_samples["all_locations"].tolist()
eb_annotated_locations[1]

[{'name': 'KASIMOW', 'start': 0, 'end': 0},
 {'name': 'Riasan', 'start': 31, 'end': 37},
 {'name': 'Babinka', 'start': 110, 'end': 117},
 {'name': 'Oka', 'start': 127, 'end': 130}]

In [98]:
eb_geo_spacy_df["all_locations"] = eb_geo_spacy_df.apply(get_all_locations_for_sample, axis=1)
eb_geo_eg_df["all_locations"] = eb_geo_eg_df.apply(get_all_locations_for_sample, axis=1)
eb_geo_flair_df["all_locations"] = eb_geo_flair_df.apply(get_all_locations_for_sample, axis=1)
eb_geo_stanza_df["all_locations"] = eb_geo_stanza_df.apply(get_all_locations_for_sample, axis=1)

In [99]:
eb_spacy_locations = eb_geo_spacy_df["all_locations"].tolist()
eb_eg_locations = eb_geo_eg_df["all_locations"].tolist()
eb_flair_locations = eb_geo_flair_df["all_locations"].tolist()
eb_stanza_locations = eb_geo_stanza_df["all_locations"].tolist()

## Evaluation

Matrix:
1. total locations count
2. Precision
3. Recall
4. F1 Score


### Total Locations Count

In [100]:
def location_count(geo_article):
    count = len(geo_article['all_locations'])
    return count

In [101]:
eb_geo_stanza_df['location_count'] = eb_geo_stanza_df.apply(location_count, axis=1)
total_locations_stanza = eb_geo_stanza_df['location_count'].sum()
print(total_locations_stanza)

40422


In [102]:
eb_geo_spacy_df['location_count'] = eb_geo_spacy_df.apply(location_count, axis=1)
total_locations_spacy = eb_geo_spacy_df['location_count'].sum()
print(total_locations_spacy)

29616


In [103]:
eb_geo_eg_df['location_count'] = eb_geo_eg_df.apply(location_count, axis=1)
total_locations_eg = eb_geo_eg_df['location_count'].sum()
print(total_locations_eg)

34363


In [104]:
eb_geo_flair_df['location_count'] = eb_geo_flair_df.apply(location_count, axis=1)
total_locations_flair = eb_geo_flair_df['location_count'].sum()
print(total_locations_flair)

40817


In [105]:
def location_found(target_location, source_list):
    """
    This function checks if the target location can be found in the source list. If there is overlapping or partial matches, it will be 1/2, when fully matching, returns 1, otherwise 0.
    :param target_location:
    :param source_list:
    :return: 1/2 when partial matching, 1 when fully matching, otherwise 0.
    """
    target_name = target_location['name']
    target_start = target_location['start']
    target_end = target_location['end']
    for location in source_list:
        if location['start'] > target_end or location['end'] < target_start:
            continue
        if target_name == location['name']:
            return 1
        return 1/2
    return 0

test_target_location = {"name":"Lake Maggiore","start":64,"end":77}
test_source_list = [{"name":"Maggiore","start":69,"end":77}, {"name":"Tyre","start":84,"end":88}]
location_found(test_target_location, test_source_list)

0.5

In [106]:
def get_tp_fp_fn(ground_truths, predictions):
    # number of locations are found in annotated samples
    # that's to say, the number of locations in predictions appears in ground_truths
    # format of locations: [{"name":"Sardinia","start":25,"end":33},....]}
    true_positive = 0
    false_positive = 0
    for target_location in predictions:
        if location_found(target_location, ground_truths) == 1:
            true_positive += 1
        elif location_found(target_location, ground_truths) == 0:
            false_positive += 1
        else:
            false_positive += 1/2

    false_negative = 0
    for target_location in ground_truths:
        if location_found(target_location, predictions) == 0:
            # print locations which are not recognised
            print(target_location)
            false_negative += 1
        elif location_found(target_location, predictions) == 1/2:
            false_negative += 1/2
    return true_positive, false_positive, false_negative

In [107]:
def get_metrix(all_annotated_locations, all_predicted_locations):
    total_true_positives = 0
    total_false_positives = 0
    total_false_negatives = 0
    for index, ground_truths in enumerate(all_annotated_locations):
        print(f"index: {index} --------")
        predictions = all_predicted_locations[index]
        true_positive, false_positive, false_negative = get_tp_fp_fn(ground_truths, predictions)
        #print(true_positive, false_positive, false_negative)
        total_true_positives += true_positive
        total_false_positives += false_positive
        total_false_negatives += false_negative
    precision = total_true_positives / (total_true_positives + total_false_positives)
    recall = total_true_positives / (total_true_positives + total_false_negatives)
    f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

In [49]:
eb_annotated_locations[15]

[{'name': 'ISSUS', 'start': 0, 'end': 0},
 {'name': 'Ajazo', 'start': 4, 'end': 9},
 {'name': 'Cilicia', 'start': 21, 'end': 28},
 {'name': 'Natolia', 'start': 32, 'end': 39},
 {'name': 'the Levant sea', 'start': 59, 'end': 73},
 {'name': 'Scanderoon', 'start': 100, 'end': 110}]

In [50]:
eb_stanza_locations[15]

[{'name': 'Ajazo', 'start': 4, 'end': 9},
 {'name': 'Cilicia', 'start': 21, 'end': 28},
 {'name': 'Natolia', 'start': 32, 'end': 39},
 {'name': 'Levant', 'start': 63, 'end': 69},
 {'name': 'Scanderoon', 'start': 100, 'end': 110}]

In [108]:
eg_metrix = get_metrix(eb_annotated_locations, eb_eg_locations[:len(eb_annotated_locations)])

index: 0 --------
index: 1 --------
{'name': 'KASIMOW', 'start': 0, 'end': 0}
{'name': 'Riasan', 'start': 31, 'end': 37}
index: 2 --------
{'name': 'INTRASCA', 'start': 0, 'end': 0}
index: 3 --------
index: 4 --------
{'name': 'KAUFBEUERN', 'start': 0, 'end': 0}
{'name': 'the Upper Danube', 'start': 38, 'end': 54}
{'name': 'Bertoch', 'start': 82, 'end': 89}
index: 5 --------
{'name': 'KEDES', 'start': 0, 'end': 0}
{'name': 'Naphtali', 'start': 55, 'end': 63}
{'name': 'Tyre', 'start': 84, 'end': 88}
{'name': 'Tyre', 'start': 182, 'end': 186}
{'name': 'Paneas', 'start': 193, 'end': 199}
{'name': 'Cidis-sus', 'start': 212, 'end': 221}
index: 6 --------
{'name': 'ILISSUS', 'start': 0, 'end': 0}
{'name': 'Eridanus', 'start': 55, 'end': 63}
{'name': 'Iliαssi-des', 'start': 152, 'end': 163}
index: 7 --------
index: 8 --------
{'name': 'KISHENGUNGA', 'start': 0, 'end': 0}
index: 9 --------
{'name': 'Roman', 'start': 257, 'end': 262}
index: 10 --------
{'name': 'Nepaul', 'start': 181, 'end': 18

In [109]:
stanza_metrix = get_metrix(eb_annotated_locations, eb_stanza_locations[:len(eb_annotated_locations)])

index: 0 --------
index: 1 --------
index: 2 --------
index: 3 --------
index: 4 --------
index: 5 --------
{'name': 'KEDES', 'start': 0, 'end': 0}
index: 6 --------
index: 7 --------
index: 8 --------
index: 9 --------
{'name': 'Roman', 'start': 257, 'end': 262}
index: 10 --------
index: 11 --------
index: 12 --------
{'name': 'INN', 'start': 0, 'end': 0}
index: 13 --------
index: 14 --------
index: 15 --------
{'name': 'ISSUS', 'start': 0, 'end': 0}
index: 16 --------
index: 17 --------
index: 18 --------
index: 19 --------
index: 20 --------
index: 21 --------
index: 22 --------
index: 23 --------
index: 24 --------
{'name': 'JAFNAPATAM', 'start': 0, 'end': 0}
index: 25 --------
index: 26 --------
index: 27 --------
index: 28 --------
index: 29 --------
index: 30 --------
index: 31 --------
index: 32 --------
index: 33 --------
{'name': 'JUIST', 'start': 0, 'end': 0}
index: 34 --------
index: 35 --------
index: 36 --------
index: 37 --------
{'name': 'KINZIG', 'start': 0, 'end': 0}


In [110]:
flair_metrix = get_metrix(eb_annotated_locations, eb_flair_locations[:len(eb_annotated_locations)])

index: 0 --------
index: 1 --------
index: 2 --------
index: 3 --------
index: 4 --------
index: 5 --------
{'name': 'KEDES', 'start': 0, 'end': 0}
{'name': 'Naphtali', 'start': 55, 'end': 63}
{'name': 'Tyre', 'start': 84, 'end': 88}
{'name': 'Galilee', 'start': 96, 'end': 103}
{'name': 'Tyre', 'start': 182, 'end': 186}
{'name': 'Paneas', 'start': 193, 'end': 199}
{'name': 'Cidis-sus', 'start': 212, 'end': 221}
{'name': 'Assyria', 'start': 251, 'end': 258}
index: 6 --------
{'name': 'ILISSUS', 'start': 0, 'end': 0}
{'name': 'Athens', 'start': 23, 'end': 29}
{'name': 'Eridanus', 'start': 55, 'end': 63}
{'name': 'Iliαssi-des', 'start': 152, 'end': 163}
index: 7 --------
index: 8 --------
index: 9 --------
{'name': 'Roman', 'start': 257, 'end': 262}
index: 10 --------
index: 11 --------
index: 12 --------
{'name': 'INN', 'start': 0, 'end': 0}
index: 13 --------
index: 14 --------
index: 15 --------
index: 16 --------
index: 17 --------
index: 18 --------
index: 19 --------
index: 20 -----

In [111]:
spacy_metrix = get_metrix(eb_annotated_locations, eb_spacy_locations[:len(eb_annotated_locations)])
eg_metrix = get_metrix(eb_annotated_locations, eb_eg_locations[:len(eb_annotated_locations)])
flair_metrix = get_metrix(eb_annotated_locations, eb_flair_locations[:len(eb_annotated_locations)])
stanza_metrix = get_metrix(eb_annotated_locations, eb_stanza_locations[:len(eb_annotated_locations)])
print(f"Spacy precision: {spacy_metrix[0]}, Edinburgh Geoparser precision: {eg_metrix[0]}, Flair precision: {flair_metrix[0]}, Stanza precision: {stanza_metrix[0]}")
print(f"Spacy recall: {spacy_metrix[1]}, Edinburgh Geoparser recall: {eg_metrix[1]}, Flair recall: {flair_metrix[1]}, Stanza recall: {stanza_metrix[1]}")
print(f"Spacy F1: {spacy_metrix[2]}, Edinburgh Geoparser f1: {eg_metrix[2]}, Flair f1: {flair_metrix[2]}, Stanza f1: {stanza_metrix[2]}")

index: 0 --------
index: 1 --------
{'name': 'KASIMOW', 'start': 0, 'end': 0}
{'name': 'Riasan', 'start': 31, 'end': 37}
{'name': 'Babinka', 'start': 110, 'end': 117}
{'name': 'Oka', 'start': 127, 'end': 130}
index: 2 --------
{'name': 'INTRASCA', 'start': 0, 'end': 0}
{'name': 'Pallanza', 'start': 51, 'end': 59}
index: 3 --------
index: 4 --------
{'name': 'KAUFBEUERN', 'start': 0, 'end': 0}
{'name': 'Bertoch', 'start': 82, 'end': 89}
index: 5 --------
{'name': 'KEDES', 'start': 0, 'end': 0}
{'name': 'Cidis-sus', 'start': 212, 'end': 221}
index: 6 --------
{'name': 'ILISSUS', 'start': 0, 'end': 0}
{'name': 'Iliαssi-des', 'start': 152, 'end': 163}
index: 7 --------
index: 8 --------
index: 9 --------
{'name': 'Roman', 'start': 257, 'end': 262}
index: 10 --------
{'name': 'Nepaul', 'start': 181, 'end': 187}
{'name': 'Nepaul', 'start': 243, 'end': 249}
{'name': 'Chinnochin', 'start': 461, 'end': 471}
index: 11 --------
{'name': 'Ossulton', 'start': 66, 'end': 74}
{'name': 'Holland House'

## Prepare data for manual annotation

In [8]:
eb_geotagged = eb_geo_stanza_df[['term_uri', 'name', 'hq_text', 'is_location', 'locations']].to_dict('records')
eb_geotagged[1]

{'term_uri': 'https://w3id.org/hto/ArticleTermRecord/9910796273804340_192693199_1007419699_0',
 'name': 'KASIMOW',
 'hq_text': 'a city of the Russian province Riasan, the capital of a circle of the same name, at the junction of the river Babinka with the Oka. It is surrounded with walls, which have been recently converted into pleasant promenades. It has a suburb inhabited by a tribe of Tartars, amounting to 500 persons, having been once the capital of a prince of that people. It contains ten churches, 1800 houses mostly of wood, and 9840 inhabitants. It has considerable trade, chiefly with the Tartars, in furs, and some manufactures of cloth and of earthen ware. Long. 4L ',
 'is_location': True,
 'locations': [{'start': 31, 'end': 37, 'name': 'Riasan'},
  {'start': 110, 'end': 117, 'name': 'Babinka'},
  {'start': 127, 'end': 130, 'name': 'Oka'}]}

In [9]:
for article in eb_geotagged:
    locations_with_tags = []
    for location in article['locations']:
        locations_with_tags.append({
            'name': location['name'],
            'start': location['start'],
            'end': location['end'],
        })
    article['locations'] = locations_with_tags
eb_geotagged[1]

{'term_uri': 'https://w3id.org/hto/ArticleTermRecord/9910796273804340_192693199_1007419699_0',
 'name': 'KASIMOW',
 'hq_text': 'a city of the Russian province Riasan, the capital of a circle of the same name, at the junction of the river Babinka with the Oka. It is surrounded with walls, which have been recently converted into pleasant promenades. It has a suburb inhabited by a tribe of Tartars, amounting to 500 persons, having been once the capital of a prince of that people. It contains ten churches, 1800 houses mostly of wood, and 9840 inhabitants. It has considerable trade, chiefly with the Tartars, in furs, and some manufactures of cloth and of earthen ware. Long. 4L ',
 'is_location': True,
 'locations': [{'name': 'Riasan', 'start': 31, 'end': 37},
  {'name': 'Babinka', 'start': 110, 'end': 117},
  {'name': 'Oka', 'start': 127, 'end': 130}]}

In [10]:
sample_location = eb_geotagged[1]['locations'][2]
eb_geotagged[1]['hq_text'][sample_location['start']:sample_location['end']]

'Oka'

In [95]:
eb_geotagged[83]['hq_text'][754:783]

'the mourning of the Egyptians'

In [25]:
eb_geotagged[11]['hq_text']

'a large parish, whose boundary joins to London, in the hundred of Ossulton, in the county of Middlesex. It is chiefly remarkable for the royal palace and gardens, the latter of which forms the scene of the recreation of the inhabitants of the metropolis. The palace is an irregular brick building, purchased by William the Third, and by him and his successors enlarged in various styles. Though with little exterior taste, it has good suites of apartments allotted to junior members of the royal family, and to other persons. The gardens have received all the advantages of which their situation is capable, and form a most pleasing promenade. Holland House and other mansions are objects of remark within this parish, as well as the numerous smaller buildings erected within the last twenty years. The inhabitants amounted in 1801 to 8556, in 1811 to 10,386, in 1821 to 14,428, and in 1831 to 20,902.'

In [9]:
eb_geotagged_df = pd.DataFrame(eb_geotagged)
eb_geotagged_df.head()

,term_uri,name,hq_text,is_location,locations
0,https://w3id.org/hto/ArticleTermRecord/9910796...,IMPEDIMENTS,"in Law, are such hindrances as put a stop or s...",False,[]
1,https://w3id.org/hto/ArticleTermRecord/9910796...,KASIMOW,"a city of the Russian province Riasan, the cap...",True,"[{'name': 'Riasan', 'start': 31, 'end': 37}, {..."
2,https://w3id.org/hto/ArticleTermRecord/9910796...,INTRASCA,"a city of the kingdom of Sardinia, in the prov...",True,"[{'name': 'Sardinia', 'start': 25, 'end': 33},..."
3,https://w3id.org/hto/ArticleTermRecord/9910796...,ILLUMINATI,the name of a secret society or order in Germa...,False,"[{'name': 'Germany', 'start': 41, 'end': 48}, ..."
4,https://w3id.org/hto/ArticleTermRecord/9910796...,KAUFBEUERN,"a city of Bavaria, in the province of the Uppe...",True,"[{'name': 'Bavaria', 'start': 10, 'end': 17}, ..."


In [11]:
eb_geotagged_df.to_json('eb_geo_samples_tagged_stanza.json', orient='records', lines=True)

In [12]:
eb_geotagged_df.iloc[:100].to_json('eb_geo_samples_annotated.json', orient='records', lines=True)

Validate data

In [145]:
import pandas as pd
eb_geo_stanza_df = pd.read_json('eb7_refined_geotagged_articles_stanza_df.json', orient='records', lines=True)

In [147]:
# all geotagged locations should have start and end index
new_locations = []
count = 0
for index, row in eb_geo_stanza_df.iterrows():
    locations = row['locations']
    if len(locations) > 0 and locations[0]["start"] < 0:
        count += 1
        print(f"error in {index}; start: {locations[0]['start']}, end: {locations[0]['end']}, name: {locations[0]['name']}")
        locations.pop(0)
    new_locations.append(locations)

print(count)

error in 10719; start: -11, end: -2, name: verschnei
error in 17284; start: -10, end: -2, name: chainwah
error in 17386; start: -8, end: -2, name: zahara
error in 20632; start: -10, end: 6, name: san juan, de los
4


In [144]:
eb_geo_stanza_df.loc[10719]['name']

'PIASANSKOI, VERSCHNEI'

In [134]:
eb_geo_stanza_df['locations'] = new_locations

In [135]:
eb_geo_stanza_df.to_json('eb1_1771_refined2_geotagged_articles_stanza_df.json', orient='records', lines=True)